In [ ]:
#Descarga red viaria y núcleos urbanos (Overpass API) a Landing

from tfm_mobility.ingesters.osm_ingester import OSMIngester

print("[BATCH INGESTION] Consultando red viaria y núcleos poblados en OpenStreetMap...")
osm_file_path = OSMIngester().fetch_batch_to_landing()

if osm_file_path:
    print(f"[OK] Datos de OSM guardados en: {osm_file_path}")
    mssparkutils.notebook.exit(osm_file_path)
else:
    raise Exception("Fallo en la ingesta de OpenStreetMap.")

In [ ]:
import os
import json
import requests
from datetime import datetime

print("🚀 [PASO 1] Iniciando extracción de la red viaria principal de España...")

# Query optimizada para autopistas/autovías (red principal nacional)
query = """
[out:json][timeout:60];
(
  way["highway"~"motorway|trunk"](35.8,-9.3,43.8,3.3);
);
out body qt 5000;
"""

headers = {
    "User-Agent": "TFMMobilityApp/1.0 (contact: pasobrad@ucm.es)",
    "Accept": "application/json"
}

# Servidores espejo de Overpass API
endpoints = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
]

data = None
for url in endpoints:
    print(f"📡 [PASO 2] Intentando conectar con: {url}")
    try:
        response = requests.post(url, data={"data": query}, headers=headers, timeout=65)
        print(f"📥 Código HTTP devuelto: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            total_tramos = len(data.get("elements", []))
            print(f"✅ ¡Éxito! Se han extraído {total_tramos} tramos de carreteras principales.")
            break
        else:
            print(f"⚠️ El servidor respondió con error: {response.status_code}")
    except Exception as e:
        print(f"❌ Falló el servidor {url}: {e}")

if data:
    print("💾 [PASO 3] Guardando resultado en Landing Batch...")
    landing_dir = "/lakehouse/default/Files/landing/batch/osm_roads"
    os.makedirs(landing_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = os.path.join(landing_dir, f"osm_roads_{timestamp}.json")
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
        
    print(f"🎉 ¡COMPLETADO! Archivo escrito correctamente en: {file_path}")
else:
    raise Exception("❌ No se pudieron descargar datos de ningún servidor de Overpass.")